**Cell #01**

# RAG11 — Stage 2 (Yoga-Sūtra): Ask Smart Questions, Some in Devanagari

Runs five hand-picked questions about the **Yoga-Sūtra of Patañjali and the Yoga-Bhāṣya** through the same
retrieval + generation pipeline as the nutrition notebooks, but against the one source that holds them:
Michel Angot's French edition (`Yogasutra.janvier. 2020.pdf, éd. 2021.pdf`, chunked by
`stage1_1_eda_packages/source18_yoga_sutra_angot.py`).

What is different from the nutrition notebooks:

1. **One source only.** `ask_question(filter_owner=...)` restricts every retrieval step to this book's
   `rag11_data_sources.rowGUID`, so nutrition chunks can't compete.
2. **A Yoga-specific system prompt** (`ask_question(system_prompt=...)`) instead of the nutrition one.
3. **Devanagari questions.** The book stores Sanskrit as IAST transliteration inside French prose
   (`yogaś cittavṛttinirodhaḥ`), so a question typed as `योगश्चित्तवृत्तिनिरोधः` shares no characters with any chunk.
   `romanize_devanagari()` (in `reusable_code/devanagari.py`) rewrites every Devanagari run into IAST and the
   notebook appends that to the question: retrieval gets something to match, the model still sees the original.
4. **No parent expansion.** In this book a "parent" is one whole sūtra with its Bhāṣya and Angot's notes
   (often 10 000+ characters), and the excerpt would only show its beginning. The 400-token child chunks
   are the precise units here, so `expand_to_parents=False`.

**Before running this notebook**: `stage1_1` must have chunked this book with
`MAX_NUMBER_OF_PAGES_TO_USE = None` (with the default cap of 100 pages the sūtra text itself, which starts on
page 244, is empty) and `stage1_2` must have loaded it. Cell #04 checks this and tells you what is missing.

In [1]:
# Cell #02
from reusable_code import (
    init_clients, ask_question, contains_devanagari, romanize_devanagari, GENERATION_MODEL,
)
from reusable_code.env import optional_env

clients = init_clients()
print("Clients ready. Supabase project:", optional_env("PUBLIC_SUPABASE_URL"), "| model:", GENERATION_MODEL)

Clients ready. Supabase project: https://czgrxgzdmodkkmbmraub.supabase.co | model: claude-sonnet-5


**Cell #03**

## Find the book in the database and check it is fully loaded

Looks the source up by file name (its `source_key` can change if files are added to the Drive folder),
then counts its child chunks and its per-sūtra parent sections. The full book has 195 sūtra sections.

In [2]:
# Cell #04
ys_rows = (clients.supabase.table("rag11_data_sources").select('"rowGUID",source_key,filename')
           .ilike("filename", "%Yogasutra%").execute().data)
if not ys_rows:
    raise RuntimeError("The Yoga-Sutra book is not in rag11_data_sources -- run stage1_1 and stage1_2 first.")

YS_OWNER_GUID = ys_rows[0]["rowGUID"]

n_children = (clients.supabase.table("rag11_chunks_child_table").select('"rowGUID"', count="exact")
              .eq("rowOwnerGUID", YS_OWNER_GUID).limit(1).execute().count)
n_sutra_parents = (clients.supabase.table("rag11_chunks_parent_table").select('"rowGUID"', count="exact")
                   .eq("rowOwnerGUID", YS_OWNER_GUID).like("title", "Yoga-S%tra%").limit(1).execute().count)

print(f"{ys_rows[0]['source_key']}: {n_children} child chunks, {n_sutra_parents} of 195 sutra sections loaded")
if n_sutra_parents < 195:
    print("\nWARNING: the sutra text is not (fully) loaded, so questions about individual sutras will get "
          "'the excerpts do not contain...' answers.\n"
          "  1. stage1_1_extract_and_chunk.ipynb: set MAX_NUMBER_OF_PAGES_TO_USE = None and re-run\n"
          "  2. stage1_2_eda_load_chunks.ipynb: re-run (only new/changed chunks are embedded)\n"
          "  3. stage1_9_eda_verify_all_data.ipynb: confirm PASS")

source18: 314 child chunks, 0 of 195 sutra sections loaded

  1. stage1_1_extract_and_chunk.ipynb: set MAX_NUMBER_OF_PAGES_TO_USE = None and re-run
  2. stage1_2_eda_load_chunks.ipynb: re-run (only new/changed chunks are embedded)
  3. stage1_9_eda_verify_all_data.ipynb: confirm PASS


**Cell #05**

## The Yoga-specific system prompt

Same contract as the nutrition prompt (answer only from the numbered excerpts; a `Short answer: Yes|No` line
only when the question has a clean yes/no answer) plus what a reader of *this* book needs: who is speaking
(sūtra, Bhāṣya, or Angot), sūtra references, IAST for Sanskrit, and the answer language.

In [3]:
# Cell #06
YS_SYSTEM_PROMPT = """You are a research assistant for a French scholarly edition of the Yoga-Sutra of \
Patanjali and the Yoga-Bhasya of Vyasa (Michel Angot, 3rd edition 2021). Answer strictly using the numbered \
excerpts in the user message -- do not rely on outside knowledge, and say plainly if the excerpts don't \
contain enough information to answer.

The excerpts are mostly French; Sanskrit appears in IAST transliteration. The question may be in English, \
French, or Devanagari (Sanskrit or Hindi); a line "[IAST: ...]" after the question is its transliteration, \
added to help retrieval. Answer in the language of the question (English for a Sanskrit-only question).

Whenever the excerpts allow it: give the sutra reference (e.g. II.35), quote the key Sanskrit term in IAST, \
and say whether a claim comes from the Sutra itself, from the Bhasya, or from Angot's own commentary.

First decide whether the question is a Yes/No question, i.e. it is worded "is/does/can/are ... ?" (or "क्या ...?") and \
a plain yes or no truly answers it. Questions that ask "what", "how", "why", "who", "which", or "what happens" are \
NEVER Yes/No questions, however the excerpts turn out.
  - If it is a Yes/No question, begin your reply with exactly this one line:
        Short answer: Yes
    or
        Short answer: No
    then a blank line, then the full explanation.
  - Otherwise skip the "Short answer" line and give the full explanation.
  - If the excerpts do not contain enough to answer, never write a "Short answer" line: say what is missing.

Keep the explanation grounded in the excerpts."""


def prepare_question(question: str) -> str:
    """Append an IAST rendering of any Devanagari in the question (kept alongside the original)."""
    if not contains_devanagari(question):
        return question
    return f"{question}\n[IAST: {romanize_devanagari(question)}]"


def format_pages(pages: list) -> str:
    """[3, 4, 5, 13, 14] -> '3-5, 13-14' (a chunk's page range is its whole section's range)."""
    ranges, start = [], None
    for prev, page in zip([None] + pages, pages):
        if prev is None or page != prev + 1:
            if start is not None:
                ranges.append((start, prev))
            start = page
    if start is not None:
        ranges.append((start, pages[-1]))
    return ", ".join(f"{a}-{b}" if a != b else str(a) for a, b in ranges) or "-"


def ask_ys(question: str, **overrides) -> dict:
    options = dict(
        match_count=6,
        use_hybrid=True,            # dense + keyword: exact sutra terms (IAST) are found by the keyword half
        use_multi_query=True,       # split compound questions into sub-questions
        use_hyde=False,
        use_rerank=True,            # cross-encoder re-scores the wide candidate pool
        expand_to_parents=False,    # a parent here is a whole sutra + commentary; keep the precise child chunks
        filter_owner=YS_OWNER_GUID,
        system_prompt=YS_SYSTEM_PROMPT,
    )
    options.update(overrides)
    return ask_question(prepare_question(question), **options)

**Cell #07**

## Nicely formatted output

`show_qa()` renders each result as a card: the **Question** (original script, plus its IAST rendering if it had
Devanagari), the **Answer** as real paragraphs and lists (the model's Markdown is converted to HTML) with the
`Short answer` line turned into a Yes/No badge, and a small footer with the retrieval details. The card has
`height: auto`, so the whole answer is always shown, never cut off or put in an inner scroll box.

(If your Jupyter front end shows a long output cell in a scrolling box, right-click the output and choose
"Disable Scrolling for Outputs", or use JupyterLab / PyCharm, which never do.)

In [4]:
# Cell #08
import html
import re

from IPython.display import HTML, display

_CSS = """<style>
.ys-card{border:1px solid rgba(128,128,128,.35);border-radius:12px;padding:16px 20px;margin:18px 0;height:auto;
  max-height:none;overflow:visible;line-height:1.6;font-size:14.5px;max-width:980px}
.ys-label{font-size:11px;letter-spacing:.09em;text-transform:uppercase;font-weight:700;opacity:.6;margin:0 0 5px}
.ys-q{font-size:15.5px;font-weight:600;margin:0 0 4px;padding:9px 13px;border-left:4px solid #2a78d6;
  background:rgba(42,120,214,.10);border-radius:4px;white-space:pre-wrap;overflow-wrap:anywhere}
.ys-iast{font-size:12.5px;opacity:.7;margin:4px 0 14px 17px;overflow-wrap:anywhere}
.ys-a{margin-top:14px;overflow-wrap:anywhere}
.ys-a p{margin:0 0 .85em}
.ys-a ul{margin:0 0 .85em 1.3em;padding:0}
.ys-a li{margin:.2em 0}
.ys-a code{background:rgba(128,128,128,.18);border-radius:4px;padding:0 4px}
.ys-badge{display:inline-block;padding:3px 12px;border-radius:999px;font-weight:700;font-size:12.5px;color:#fff;margin-bottom:10px}
.ys-yes{background:#0b7a4b}.ys-no{background:#b23b2e}
.ys-meta{margin-top:12px;padding-top:9px;border-top:1px dashed rgba(128,128,128,.45);font-size:12px;opacity:.75;overflow-wrap:anywhere}
table.ys-tbl{border-collapse:collapse;font-size:13px;margin:10px 0}
table.ys-tbl th,table.ys-tbl td{border-bottom:1px solid rgba(128,128,128,.35);padding:5px 12px;text-align:left;vertical-align:top}
table.ys-tbl th{font-size:11px;text-transform:uppercase;letter-spacing:.06em;opacity:.7}
</style>"""


def _inline(text: str) -> str:
    text = html.escape(text)
    text = re.sub(r"\*\*(.+?)\*\*", r"<b>\1</b>", text)
    text = re.sub(r"(?<![*\w])\*(?!\s)(.+?)(?<!\s)\*(?![*\w])", r"<i>\1</i>", text)
    return re.sub(r"`([^`]+)`", r"<code>\1</code>", text)


def answer_html(text: str) -> str:
    """Tiny Markdown -> HTML: paragraphs, '-'/'*' bullet lists, **bold**, *italic*, `code`."""
    out, para, items = [], [], []

    def flush():
        if para:
            out.append("<p>" + "<br>".join(_inline(line) for line in para) + "</p>")
            para.clear()
        if items:
            out.append("<ul>" + "".join(f"<li>{_inline(i)}</li>" for i in items) + "</ul>")
            items.clear()

    for line in text.splitlines():
        stripped = line.strip()
        if not stripped:
            flush()
            continue
        bullet = re.match(r"^[-*•]\s+(.*)", stripped)
        if bullet:
            if para:
                flush()
            items.append(bullet.group(1))
        else:
            if items:
                flush()
            para.append(stripped)
    flush()
    return "".join(out)


_SHORT_ANSWER_LINE = re.compile(r"^\s*Short answer:\s*(Yes|No)\s*\n*", re.IGNORECASE)


def show_qa(number: int, question: str, result: dict) -> None:
    answer = _SHORT_ANSWER_LINE.sub("", result["answer"], count=1).strip()
    badge = ""
    if result["short_answer"]:
        cls = "ys-yes" if result["short_answer"] == "Yes" else "ys-no"
        badge = f'<div class="ys-badge {cls}">Short answer: {result["short_answer"]}</div>'
    iast = ""
    if contains_devanagari(question):
        iast = f'<div class="ys-iast">IAST: {html.escape(romanize_devanagari(question))}</div>'
    meta = [
        f"{result['chunks_used']} excerpts (best of {result['candidates_considered']} candidates)",
        f"pages {format_pages(result['source_pages'])}",
    ]
    if result["subquestions"] and len(result["subquestions"]) > 1:
        meta.append("sub-questions: " + " &#124; ".join(html.escape(q) for q in result["subquestions"]))
    if result["grounding_words"]:
        meta.append("established on: " + html.escape(", ".join(result["grounding_words"])))
    display(HTML(
        _CSS
        + '<div class="ys-card">'
        + f'<div class="ys-label">Question {number}:</div><div class="ys-q">{html.escape(question)}</div>{iast}'
        + '<div class="ys-a"><div class="ys-label">Answer:</div>' + badge + answer_html(answer) + "</div>"
        + '<div class="ys-meta">' + "<br>".join(meta) + "</div></div>"
    ))

**Cell #09**

## The five questions

Each targets a different part of the book and a different retrieval difficulty:

1. **Devanagari sūtra + a "how does the Bhāṣya explain" question**: the sūtra (I.2) is quoted in Devanagari, so
   only the IAST rendering can reach the chunk; the answer needs both the sūtra and the Bhāṣya's gloss of *nirodha*.
2. **Interpretive question about Angot's method** (English): the answer lives in one of his appendix notices
   (*Adhikāra I.1 et les destinataires du Yoga-Sūtra*), written in French, so this is cross-lingual retrieval.
3. **Devanagari sūtra + a consequence** (II.35): quote in Devanagari, ask what follows from it; the Bhāṣya and
   Angot's notes add nuance beyond the one-line sūtra.
4. **A multi-part comparison** (English, IAST terms): the five *vṛtti*-s and which are *kliṣṭa* / *akliṣṭa*
   (I.5-11): spread across seven sūtras, the case multi-query splitting exists for.
5. **A Hindi yes/no question written entirely in Devanagari**: *is Īśvara in the Yoga-Sūtra a creator god?* A
   clean Yes/No with an important nuance (I.23-26, II.1, II.45), and no Latin letters at all in the question.

In [5]:
# Cell #10
YS_QUESTIONS = [
    # 1. Devanagari sutra (I.2) + English question about the Bhasya's gloss of nirodha.
    "योगश्चित्तवृत्तिनिरोधः -- what does Yoga-Sūtra I.2 define yoga as, and how does the Bhāṣya explain the word nirodha?",
    # 2. Interpretive: Angot's reading of the very first word, answer is in a French appendix notice.
    "Why does Angot read the opening word 'atha' of Yoga-Sūtra I.1 as an adhikāra, and who are the intended addressees of the text?",
    # 3. Devanagari sutra (II.35) + what follows from it.
    "अहिंसाप्रतिष्ठायां तत्सन्निधौ वैरत्यागः -- according to II.35, what happens in the presence of someone firmly established in ahiṃsā?",
    # 4. Multi-part comparison across I.5-I.11 -- the case for multi-query splitting.
    "How do the five vṛtti-s (pramāṇa, viparyaya, vikalpa, nidrā, smṛti) differ from one another, and which of them can be kliṣṭa or akliṣṭa?",
    # 5. Hindi, all Devanagari, clean Yes/No with a nuance.
    "क्या योगसूत्र में ईश्वर जगत् का सृष्टिकर्ता है?",
]

**Cell #11**

## What the Devanagari questions turn into

`romanize_devanagari()` only touches Devanagari characters; Latin text, digits and punctuation stay as they are.

In [6]:
# Cell #12
for i, q in enumerate(YS_QUESTIONS, start=1):
    if contains_devanagari(q):
        print(f"Q{i} original: {q}")
        print(f"Q{i} IAST    : {romanize_devanagari(q)}\n")

Q1 original: योगश्चित्तवृत्तिनिरोधः -- what does Yoga-Sūtra I.2 define yoga as, and how does the Bhāṣya explain the word nirodha?
Q1 IAST    : yogaścittavṛttinirodhaḥ -- what does Yoga-Sūtra I.2 define yoga as, and how does the Bhāṣya explain the word nirodha?

Q3 original: अहिंसाप्रतिष्ठायां तत्सन्निधौ वैरत्यागः -- according to II.35, what happens in the presence of someone firmly established in ahiṃsā?
Q3 IAST    : ahiṃsāpratiṣṭhāyāṃ tatsannidhau vairatyāgaḥ -- according to II.35, what happens in the presence of someone firmly established in ahiṃsā?

Q5 original: क्या योगसूत्र में ईश्वर जगत् का सृष्टिकर्ता है?
Q5 IAST    : kyā yogasūtra meṃ īśvara jagat kā sṛṣṭikartā hai?



**Cell #13**

## Run all 5 questions

Each question goes through `ask_ys()` and is displayed as a Question / Answer card by `show_qa()`. The footer of each
card shows the pipeline's bookkeeping: how many candidates the reranker cut down to the final six excerpts, the
(compressed) page ranges they come from, the sub-questions multi-query produced, and the words the answer shares
with the excerpts.

In [ ]:
# Cell #14
ys_results = []
for number, question in enumerate(YS_QUESTIONS, start=1):
    result = ask_ys(question)
    ys_results.append(result)
    show_qa(number, question, result)

**Cell #15**

## Summary table

In [ ]:
# Cell #16
rows = []
for i, (q, r) in enumerate(zip(YS_QUESTIONS, ys_results), start=1):
    script = "Devanagari" if contains_devanagari(q) else "Latin"
    rows.append(
        f"<tr><td>{i}</td><td>{script}</td><td>{r['short_answer'] or 'n/a'}</td>"
        f"<td>{len(r['subquestions'] or [])}</td><td>{r['chunks_used']}</td>"
        f"<td>{html.escape(q[:90])}{'...' if len(q) > 90 else ''}</td></tr>"
    )
display(HTML(
    _CSS + '<table class="ys-tbl"><tr><th>#</th><th>script</th><th>short answer</th><th>sub-Qs</th>'
    '<th>excerpts</th><th>question</th></tr>' + "".join(rows) + "</table>"
))

**Cell #17**

## What does the IAST rendering buy? (retrieval only)

Multi-query splitting can hide the effect, because the model that splits the question often transliterates the
Devanagari itself (look at the sub-questions printed above). So this cell compares **hybrid retrieval alone**
for question 3 with and without the appended `[IAST: ...]` line: the top chunks and which half of the hybrid
search (dense or keyword) found each one. Once the sūtra text is fully loaded, the IAST version should surface
the II.35 section; the raw Devanagari version has no letters in common with any chunk.

In [ ]:
# Cell #18
from reusable_code import hybrid_search

question = YS_QUESTIONS[2]
for label, text in [("raw Devanagari", question), ("with [IAST: ...]", prepare_question(question))]:
    rows = hybrid_search(text, match_count=5, filter_owner=YS_OWNER_GUID)
    print(f"--- {label}")
    for r in rows:
        first_line = r["rowJSON"]["text"].split("\n", 1)[0][:95]
        print(f"  dense#{r['dense_rank']!s:<4} keyword#{r['keyword_rank']!s:<5} {first_line}")

**Cell #19**

## Save workspace to GitHub

Synchronize this notebook and any code changes to GitHub (with auto lock recovery and conflict resolution).

In [ ]:
# Cell #20
from reusable_code import save_to_github

save_to_github("stage2_ask_examples7_ys.ipynb - Yoga-Sutra questions incl. Devanagari")